# CSCI 567 — Pre-final: Temporally Robust and Fair Credit Risk Prediction
**LendingClub 2007–2018**

- Exp 1: Temporal Robustness — temporal vs random split AUC drop
- Exp 2: Model Design Effects — hyperparameter sweep
- Exp 3: Fairness Analysis — credit history group + income group
- Exp 4: Robustness–Fairness Trade-off
- Exp 5: Neural Network
- Annual AUC trend (2015, 2016, 2017, 2018)

## 0. Setup Enviroment & Data Load

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!cp "/content/drive/MyDrive/accepted_2007_to_2018Q4.csv.gz" /content/

In [6]:
import os
os.listdir('/content/')

['.config', 'accepted_2007_to_2018Q4.csv.gz', 'drive', 'sample_data']

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('/content/accepted_2007_to_2018Q4.csv.gz', low_memory=False)

print(f'Raw shape: {df.shape}')

Raw shape: (2260701, 151)


## 1. Preprocessing

In [8]:

# ── Target Creation ──────────────────────────────────────────────────────────
default_labels = ['Charged Off', 'Default',
                  'Does not meet the credit policy. Status:Charged Off']
df['target'] = df['loan_status'].isin(default_labels).astype(int)

# ── Date Parsing ──────────────────────────────────────────────────────────
df['issue_d']  = pd.to_datetime(df['issue_d'],         format='%b-%Y')
df['issue_year'] = df['issue_d'].dt.year
df['age_proxy'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y')
df['credit_history_years'] = (df['issue_d'] - df['age_proxy']).dt.days / 365

# ── Feature Selection ──────────────────────────────────────────────────────────
FEATURES = ['loan_amnt', 'int_rate', 'installment', 'annual_inc',
            'dti', 'delinq_2yrs', 'fico_range_low', 'open_acc',
            'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
            'mort_acc', 'pub_rec_bankruptcies']

# ── Handling Missing Values ────────────────────────────────────────────────────────
df[FEATURES] = df[FEATURES].fillna(df[FEATURES].median())
df['credit_history_years'] = df['credit_history_years'].fillna(
    df['credit_history_years'].median())

# ── Sampling (Colab) ──────────────────────────────────────────────
df_sample = df.sample(n=200000, random_state=42).copy()
# df_sample = df.copy()
print(f'Sample shape: {df_sample.shape}')
print(f'Default rate: {df_sample["target"].mean():.3f}')

Sample shape: (200000, 155)
Default rate: 0.119


In [9]:
# ── Fairness group creation ─────────────────────────────────────────────────
# 1) Credit history length (same as check in)
df_sample['credit_group'] = pd.qcut(
    df_sample['credit_history_years'], q=3,
    labels=['short', 'mid', 'long'])

# 2) Income group (add Pre-final)
df_sample['income_group'] = pd.qcut(
    df_sample['annual_inc'].clip(upper=df_sample['annual_inc'].quantile(0.99)),
    q=3, labels=['low', 'mid', 'high'])

print('Credit group distribution:')
print(df_sample['credit_group'].value_counts())
print('\nIncome group distribution:')
print(df_sample['income_group'].value_counts())

Credit group distribution:
credit_group
short    67120
long     66582
mid      66298
Name: count, dtype: int64

Income group distribution:
income_group
low     67693
high    66434
mid     65873
Name: count, dtype: int64


## 2. Data Splitting

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ── Temporal split: train 2007-2014 / test 2015-2018 ──────────────────
train_t = df_sample[df_sample['issue_year'] <= 2014]
test_t  = df_sample[df_sample['issue_year'] >= 2015]

X_train_t = train_t[FEATURES]
y_train_t = train_t['target']
X_test_t  = test_t[FEATURES]
y_test_t  = test_t['target']

# ── Random split (for comparision) ──────────────────────────────────────────────
X_rand_tr, X_rand_te, y_rand_tr, y_rand_te = train_test_split(
    df_sample[FEATURES], df_sample['target'],
    test_size=0.2, random_state=42)

# ── Scaling ────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_t)
X_test_sc  = scaler.transform(X_test_t)
X_rand_tr_sc = scaler.transform(X_rand_tr)
X_rand_te_sc = scaler.transform(X_rand_te)

print(f'Temporal  — Train: {len(X_train_t):,}  Test: {len(X_test_t):,}')
print(f'Random    — Train: {len(X_rand_tr):,}  Test: {len(X_rand_te):,}')

Temporal  — Train: 41,242  Test: 158,754
Random    — Train: 160,000  Test: 40,000


In [11]:
# ── By year test set (AUC over time) ─────────────────────────────────
yearly_tests = {}
for yr in [2015, 2016, 2017, 2018]:
    sub = df_sample[df_sample['issue_year'] == yr]
    if len(sub) > 0:
        yearly_tests[yr] = (sub[FEATURES], sub['target'],
                            scaler.transform(sub[FEATURES]),
                            sub)
        print(f'{yr}: {len(sub):,} samples, default rate={sub["target"].mean():.3f}')

2015: 37,158 samples, default rate=0.183
2016: 38,347 samples, default rate=0.156
2017: 39,269 samples, default rate=0.090
2018: 43,980 samples, default rate=0.017


## 3. Model Definition — Full Model + Hyperparameter Sweep


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

# ── All Model Definitions ─────────────────────────────────────────────────────
# LR: C Value sweep (regularization strength)
# Tree: max_depth sweep
# RF: n_estimators x max_depth
# XGB: learning_rate x max_depth
# NN: hidden layer size x alpha (L2)

MODEL_CONFIGS = {
    # Logistic Regression — C sweep
    'LR_C0.001':  (LogisticRegression(C=0.001, max_iter=1000), 'scaled'),
    'LR_C0.01':   (LogisticRegression(C=0.01,  max_iter=1000), 'scaled'),
    'LR_C0.1':    (LogisticRegression(C=0.1,   max_iter=1000), 'scaled'),
    'LR_C1':      (LogisticRegression(C=1,     max_iter=1000), 'scaled'),
    'LR_C10':     (LogisticRegression(C=10,    max_iter=1000), 'scaled'),

    # Decision Tree — depth sweep
    'DTree_d3':   (DecisionTreeClassifier(max_depth=3,  random_state=42), 'raw'),
    'DTree_d5':   (DecisionTreeClassifier(max_depth=5,  random_state=42), 'raw'),
    'DTree_d7': (DecisionTreeClassifier(max_depth=7, random_state=42), "raw"),
    'DTree_d10':  (DecisionTreeClassifier(max_depth=10, random_state=42), 'raw'),

    # Random Forest — n_estimators x depth
    'RF_50_d5':   (RandomForestClassifier(n_estimators=50,  max_depth=5,  random_state=42, n_jobs=-1), 'raw'),
    'RF_100_d5':  (RandomForestClassifier(n_estimators=100, max_depth=5,  random_state=42, n_jobs=-1), 'raw'),
    'RF_100_d10': (RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1), 'raw'),

    # XGBoost — lr x depth
    'XGB_lr0.01': (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=4, eval_metric='logloss', random_state=42), 'raw'),
    'XGB_lr0.1':  (XGBClassifier(n_estimators=100, learning_rate=0.1,  max_depth=4, eval_metric='logloss', random_state=42), 'raw'),
    'XGB_d6':     (XGBClassifier(n_estimators=100, learning_rate=0.1,  max_depth=6, eval_metric='logloss', random_state=42), 'raw'),

    # Neural Network — architecture x alpha sweep
    # 'NN_small':   (MLPClassifier(hidden_layer_sizes=(64,),       alpha=0.01, max_iter=200, random_state=42), 'scaled'),
    # 'NN_mid':     (MLPClassifier(hidden_layer_sizes=(128, 64),   alpha=0.01, max_iter=200, random_state=42), 'scaled'),
    # 'NN_large':   (MLPClassifier(hidden_layer_sizes=(256, 128, 64), alpha=0.01, max_iter=200, random_state=42), 'scaled'),
    # 'NN_reg':     (MLPClassifier(hidden_layer_sizes=(128, 64),   alpha=0.1,  max_iter=200, random_state=42), 'scaled'),
}

print(f'Total model configs: {len(MODEL_CONFIGS)}')

Total model configs: 15


## Decision Tree Testing

In [13]:
from sklearn.tree import DecisionTreeClassifier

MODEL_CONFIGS_DTREE = {
    # Base tree
    'DT_base':          (DecisionTreeClassifier(random_state=42), 'raw'),

    # 1) max_depth sweep
    'DT_d3':            (DecisionTreeClassifier(max_depth=3,  random_state=42), 'raw'),
    'DT_d5':            (DecisionTreeClassifier(max_depth=5,  random_state=42), 'raw'),
    'DT_d7':            (DecisionTreeClassifier(max_depth=7,  random_state=42), 'raw'),
    'DT_d9':            (DecisionTreeClassifier(max_depth=9,  random_state=42), 'raw'),

    # 2) min_samples_leaf sweep
    'DT_d7_leaf1':      (DecisionTreeClassifier(max_depth=7, min_samples_leaf=1,  random_state=42), 'raw'),
    'DT_d7_leaf20':     (DecisionTreeClassifier(max_depth=7, min_samples_leaf=20, random_state=42), 'raw'),
    'DT_d7_leaf50':     (DecisionTreeClassifier(max_depth=7, min_samples_leaf=50, random_state=42), 'raw'),

    # 3) max_features sweep
    'DT_d7_mf_all':     (DecisionTreeClassifier(max_depth=7, max_features=None,  random_state=42), 'raw'),
    'DT_d7_mf_sqrt':    (DecisionTreeClassifier(max_depth=7, max_features='sqrt', random_state=42), 'raw'),
    'DT_d7_mf_log2':    (DecisionTreeClassifier(max_depth=7, max_features='log2', random_state=42), 'raw'),
}

print(f"Total DT configs: {len(MODEL_CONFIGS_DTREE)}")

Total DT configs: 11


In [14]:
from sklearn.metrics import roc_auc_score, accuracy_score, brier_score_loss
import time

results_DTREE = {}

for name, (model, input_type) in MODEL_CONFIGS_DTREE.items():
    t0 = time.time()

    if input_type == 'scaled':
        model.fit(X_train_sc, y_train_t)
        y_prob_t = model.predict_proba(X_test_sc)[:, 1]
        y_pred_t = model.predict(X_test_sc)
        y_prob_r = model.predict_proba(X_rand_te_sc)[:, 1]
        y_pred_r = model.predict(X_rand_te_sc)
    else:
        model.fit(X_train_t, y_train_t)
        y_prob_t = model.predict_proba(X_test_t)[:, 1]
        y_pred_t = model.predict(X_test_t)
        y_prob_r = model.predict_proba(X_rand_te)[:, 1]
        y_pred_r = model.predict(X_rand_te)

    results_DTREE[name] = {
        'temporal': {
            'auc':   roc_auc_score(y_test_t, y_prob_t),
            'acc':   accuracy_score(y_test_t, y_pred_t),
            'brier': brier_score_loss(y_test_t, y_prob_t),
            'y_pred': y_pred_t,
            'y_prob': y_prob_t,
        },
        'random': {
            'auc':   roc_auc_score(y_rand_te, y_prob_r),
            'acc':   accuracy_score(y_rand_te, y_pred_r),
            'brier': brier_score_loss(y_rand_te, y_prob_r),
        },
        'yearly': {}
    }

    # yearly AUCs
    for yr, (X_yr, y_yr, X_yr_sc, _) in yearly_tests.items():
        X_in = X_yr_sc if input_type == 'scaled' else X_yr
        yp = model.predict_proba(X_in)[:, 1]
        results_DTREE[name]['yearly'][yr] = roc_auc_score(y_yr, yp)

    results_DTREE[name]['auc_drop'] = (
        results_DTREE[name]['random']['auc'] -
        results_DTREE[name]['temporal']['auc']
    )

    elapsed = time.time() - t0

In [15]:
for name, res in sorted(results_DTREE.items(), key=lambda x: x[1]['temporal']['auc'], reverse=True):
    print(f"{name:15s}  temporal_AUC={res['temporal']['auc']:.4f}  "
          f"random_AUC={res['random']['auc']:.4f}  "
          f"AUC_drop={res['auc_drop']:.4f}")

DT_d3            temporal_AUC=0.6738  random_AUC=0.6781  AUC_drop=0.0044
DT_d5            temporal_AUC=0.6727  random_AUC=0.6831  AUC_drop=0.0103
DT_d7_leaf50     temporal_AUC=0.6683  random_AUC=0.6828  AUC_drop=0.0145
DT_d7_leaf20     temporal_AUC=0.6670  random_AUC=0.6831  AUC_drop=0.0162
DT_d7            temporal_AUC=0.6659  random_AUC=0.6816  AUC_drop=0.0156
DT_d7_leaf1      temporal_AUC=0.6659  random_AUC=0.6816  AUC_drop=0.0156
DT_d7_mf_all     temporal_AUC=0.6659  random_AUC=0.6816  AUC_drop=0.0156
DT_d9            temporal_AUC=0.6521  random_AUC=0.6770  AUC_drop=0.0248
DT_d7_mf_sqrt    temporal_AUC=0.6364  random_AUC=0.6518  AUC_drop=0.0155
DT_d7_mf_log2    temporal_AUC=0.6364  random_AUC=0.6518  AUC_drop=0.0155
DT_base          temporal_AUC=0.5307  random_AUC=0.6572  AUC_drop=0.1265


In [19]:
def compute_fairness(y_true, y_pred, y_prob, groups, group_col):
    """Group FPR, TPR, AUC calculation"""
    records = []
    for g in groups:
        mask = (group_col == g).values
        yt = np.array(y_true)[mask]
        yp = np.array(y_pred)[mask]
        ypr = np.array(y_prob)[mask]

        if yt.sum() == 0 or (1 - yt).sum() == 0:
            continue

        tp = ((yp == 1) & (yt == 1)).sum()
        fp = ((yp == 1) & (yt == 0)).sum()
        tn = ((yp == 0) & (yt == 0)).sum()
        fn = ((yp == 0) & (yt == 1)).sum()

        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        auc = roc_auc_score(yt, ypr)

        records.append({
            'group': g,
            'FPR': fpr,
            'TPR': tpr,
            'AUC': auc,
            'n': mask.sum()
        })
    return pd.DataFrame(records)

In [20]:
# Fairness for Decision Trees

fairness_results_DT = {}   # {model: {'credit': df, 'income': df}}

test_meta = test_t.copy()

for name in MODEL_CONFIGS_DTREE.keys():
    y_pred_t = results_DTREE[name]['temporal']['y_pred']
    y_prob_t = results_DTREE[name]['temporal']['y_prob']

    fairness_results_DT[name] = {
        'credit': compute_fairness(
            y_test_t, y_pred_t, y_prob_t,
            ['short', 'mid', 'long'], test_meta['credit_group']
        ),
        'income': compute_fairness(
            y_test_t, y_pred_t, y_prob_t,
            ['low', 'mid', 'high'], test_meta['income_group']
        ),
    }

print('Fairness computed for DT models:', list(fairness_results_DT.keys()))

Fairness computed for DT models: ['DT_base', 'DT_d3', 'DT_d5', 'DT_d7', 'DT_d9', 'DT_d7_leaf1', 'DT_d7_leaf20', 'DT_d7_leaf50', 'DT_d7_mf_all', 'DT_d7_mf_sqrt', 'DT_d7_mf_log2']


In [22]:
# === Decision Tree Fairness Disparity Summary ===

rows = []
for name in MODEL_CONFIGS_DTREE.keys():   # only DT models
    credit_df = fairness_results_DT[name]['credit']
    income_df = fairness_results_DT[name]['income']

    rows.append({
        'Model':            name,
        'Temporal AUC':     round(results_DTREE[name]['temporal']['auc'], 4),
        'Accuracy':         round(results_DTREE[name]['temporal']['acc'], 4),
        'AUC Drop':         round(results_DTREE[name]['auc_drop'], 4),
        'Credit FPR_disp':  round(credit_df['FPR'].max() - credit_df['FPR'].min(), 4),
        'Credit TPR_disp':  round(credit_df['TPR'].max() - credit_df['TPR'].min(), 4),
        'Credit AUC_disp':  round(credit_df['AUC'].max() - credit_df['AUC'].min(), 4),
        'Income FPR_disp':  round(income_df['FPR'].max() - income_df['FPR'].min(), 4),
        'Income TPR_disp':  round(income_df['TPR'].max() - income_df['TPR'].min(), 4),
        'Income AUC_disp':  round(income_df['AUC'].max() - income_df['AUC'].min(), 4),
    })

df_fairness_summary_dt = pd.DataFrame(rows).sort_values('Temporal AUC', ascending=False)

print('\n=== Decision Tree Fairness Disparity Summary ===')
print(df_fairness_summary_dt.to_string(index=False))


=== Decision Tree Fairness Disparity Summary ===
        Model  Temporal AUC  Accuracy  AUC Drop  Credit FPR_disp  Credit TPR_disp  Credit AUC_disp  Income FPR_disp  Income TPR_disp  Income AUC_disp
        DT_d3        0.6738    0.8927    0.0044           0.0000           0.0000           0.0067           0.0000           0.0000           0.0369
        DT_d5        0.6727    0.8857    0.0103           0.0090           0.0188           0.0121           0.0242           0.0549           0.0422
 DT_d7_leaf50        0.6683    0.8897    0.0145           0.0068           0.0165           0.0141           0.0126           0.0325           0.0440
 DT_d7_leaf20        0.6670    0.8895    0.0162           0.0058           0.0119           0.0105           0.0122           0.0288           0.0337
 DT_d7_mf_all        0.6659    0.8832    0.0156           0.0031           0.0131           0.0102           0.0176           0.0330           0.0311
  DT_d7_leaf1        0.6659    0.8832    0.0156   

## XGBoost Test Code (Find Best)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

# ── All Model Definitions ─────────────────────────────────────────────────────
# LR: C Value sweep (regularization strength)
# Tree: max_depth sweep
# RF: n_estimators x max_depth
# XGB: learning_rate x max_depth
# NN: hidden layer size x alpha (L2)

MODEL_CONFIGS = {
    # 1) Baseline
    'XGB_base':         (XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, eval_metric='logloss', random_state=42), 'raw'),

    # 2) learning_rate sweep
    'XGB_lr0.01':       (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=4, eval_metric='logloss', random_state=42), 'raw'),
    'XGB_lr0.05':       (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=4, eval_metric='logloss', random_state=42), 'raw'),#best
    'XGB_lr0.3':        (XGBClassifier(n_estimators=100, learning_rate=0.3,  max_depth=4, eval_metric='logloss', random_state=42), 'raw'),

    # 3) max_depth sweep
    'XGB_d3':           (XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, eval_metric='logloss', random_state=42), 'raw'),#best
    'XGB_d6':           (XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=6, eval_metric='logloss', random_state=42), 'raw'),
    'XGB_d8':           (XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=8, eval_metric='logloss', random_state=42), 'raw'),

    # 4) min_child_weight sweep
    'XGB_mcw3':         (XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, min_child_weight=3,  eval_metric='logloss', random_state=42), 'raw'),#best
    'XGB_mcw5':         (XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, min_child_weight=5,  eval_metric='logloss', random_state=42), 'raw'),
    'XGB_mcw10':        (XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, min_child_weight=10, eval_metric='logloss', random_state=42), 'raw'),

    # 5) subsample sweep
    'XGB_sub0.6':       (XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, subsample=0.6, eval_metric='logloss', random_state=42), 'raw'),
    'XGB_sub0.7':       (XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, subsample=0.7, eval_metric='logloss', random_state=42), 'raw'),#best
    'XGB_sub0.8':       (XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, subsample=0.8, eval_metric='logloss', random_state=42), 'raw'),

    # 6) colsample_bytree sweep
    'XGB_col0.6':       (XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, colsample_bytree=0.6, eval_metric='logloss', random_state=42), 'raw'),#best
    'XGB_col0.7':       (XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, colsample_bytree=0.7, eval_metric='logloss', random_state=42), 'raw'),
    'XGB_col0.8':       (XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, colsample_bytree=0.8, eval_metric='logloss', random_state=42), 'raw'),

    # 7) gamma sweep
    'XGB_gamma1':       (XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, gamma=1, eval_metric='logloss', random_state=42), 'raw'),
    'XGB_gamma3':       (XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, gamma=3, eval_metric='logloss', random_state=42), 'raw'),#best
    'XGB_gamma5':       (XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, gamma=5, eval_metric='logloss', random_state=42), 'raw'),

    # 8) reg_alpha (L1) sweep
    'XGB_L1_0.1':       (XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, reg_alpha=0.1, eval_metric='logloss', random_state=42), 'raw'),
    'XGB_L1_1':         (XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, reg_alpha=1.0, eval_metric='logloss', random_state=42), 'raw'),
    'XGB_L1_5':         (XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, reg_alpha=5.0, eval_metric='logloss', random_state=42), 'raw'),#best

    # 9) reg_lambda (L2) sweep
    'XGB_L2_1':         (XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, reg_lambda=1.0, eval_metric='logloss', random_state=42), 'raw'),
    'XGB_L2_5':         (XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, reg_lambda=5.0, eval_metric='logloss', random_state=42), 'raw'),
    'XGB_L2_10':        (XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, reg_lambda=10.0, eval_metric='logloss', random_state=42), 'raw'),#best

    # 10) n_estimators sweep
    'XGB_n50':          (XGBClassifier(n_estimators=50,  learning_rate=0.1, max_depth=4, eval_metric='logloss', random_state=42), 'raw'),#best
    'XGB_n200':         (XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=4, eval_metric='logloss', random_state=42), 'raw'),
    'XGB_n500':         (XGBClassifier(n_estimators=500, learning_rate=0.1, max_depth=4, eval_metric='logloss', random_state=42), 'raw'),

    # 조합 — 위에서 좋았던 것들 합치기
    'XGB_combo_v1':         (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, gamma=3, eval_metric='logloss', random_state=42), 'raw'),
    'XGB_combo_v2':         (XGBClassifier(n_estimators=70, learning_rate=0.05, max_depth=3, gamma=3, eval_metric='logloss', random_state=42), 'raw'),
    'XGB_combo_v3':         (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, gamma=3, eval_metric='logloss', random_state=42), 'raw'),
    'XGB_combo_v4':         (XGBClassifier(n_estimators=70, learning_rate=0.01, max_depth=3, gamma=3, eval_metric='logloss', random_state=42), 'raw'),
}

print(f'Total model configs: {len(MODEL_CONFIGS)}')

In [ ]:
# Training & Temporal / Random AUC calculation
from sklearn.metrics import roc_auc_score, accuracy_score, brier_score_loss
import time

results = {}   # {model_name: {'temporal': {...}, 'random': {...}, 'yearly': {...}}}

for name, (model, input_type) in MODEL_CONFIGS.items():
    t0 = time.time()

    if input_type == 'scaled':
        model.fit(X_train_sc, y_train_t)
        y_prob_t  = model.predict_proba(X_test_sc)[:, 1]
        y_pred_t  = model.predict(X_test_sc)
        y_prob_r  = model.predict_proba(X_rand_te_sc)[:, 1]
        y_pred_r  = model.predict(X_rand_te_sc)
    else:
        model.fit(X_train_t, y_train_t)
        y_prob_t  = model.predict_proba(X_test_t)[:, 1]
        y_pred_t  = model.predict(X_test_t)
        y_prob_r  = model.predict_proba(X_rand_te)[:, 1]
        y_pred_r  = model.predict(X_rand_te)

    # Temporal result
    results[name] = {
        'temporal': {
            'auc':   roc_auc_score(y_test_t, y_prob_t),
            'acc':   accuracy_score(y_test_t, y_pred_t),
            'brier': brier_score_loss(y_test_t, y_prob_t),
            'y_pred': y_pred_t,
            'y_prob': y_prob_t,
        },
        'random': {
            'auc':   roc_auc_score(y_rand_te, y_prob_r),
            'acc':   accuracy_score(y_rand_te, y_pred_r),
            'brier': brier_score_loss(y_rand_te, y_prob_r),
        },
        'yearly': {}
    }

    # By year AUC
    for yr, (X_yr, y_yr, X_yr_sc, _) in yearly_tests.items():
        X_in = X_yr_sc if input_type == 'scaled' else X_yr
        yp = model.predict_proba(X_in)[:, 1]
        results[name]['yearly'][yr] = roc_auc_score(y_yr, yp)

    # AUC drop
    results[name]['auc_drop'] = results[name]['random']['auc'] - results[name]['temporal']['auc']

    elapsed = time.time() - t0
    # print(f"{name:20s}  temporal_AUC={results[name]['temporal']['auc']:.4f}  "
    #       f"random_AUC={results[name]['random']['auc']:.4f}  "
    #       f"AUC_drop={results[name]['auc_drop']:.4f}  ({elapsed:.1f}s)")


In [ ]:
#Best temporal_AUC
for name, res in sorted(results.items(), key=lambda x: x[1]['temporal']['auc'], reverse=True):
    print(f"{name:20s}  temporal_AUC={res['temporal']['auc']:.4f}  "
          f"random_AUC={res['random']['auc']:.4f}  "
          f"AUC_drop={res['auc_drop']:.4f}")

In [ ]:
#Best AUC_drop
for name, res in sorted(results.items(), key=lambda x: x[1]['auc_drop']):
    print(f"{name:20s}  temporal_AUC={res['temporal']['auc']:.4f}  "
          f"random_AUC={res['random']['auc']:.4f}  "
          f"AUC_drop={res['auc_drop']:.4f}")

In [ ]:
def compute_fairness(y_true, y_pred, y_prob, groups, group_col):
    """Group FPR, TPR, AUC calculation"""
    records = []
    for g in groups:
        mask = (group_col == g).values
        yt = np.array(y_true)[mask]
        yp = np.array(y_pred)[mask]
        ypr = np.array(y_prob)[mask]

        if yt.sum() == 0 or (1 - yt).sum() == 0:
            continue

        tp = ((yp == 1) & (yt == 1)).sum()
        fp = ((yp == 1) & (yt == 0)).sum()
        tn = ((yp == 0) & (yt == 0)).sum()
        fn = ((yp == 0) & (yt == 1)).sum()

        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        auc = roc_auc_score(yt, ypr)

        records.append({'group': g, 'FPR': fpr, 'TPR': tpr, 'AUC': auc, 'n': mask.sum()})
    return pd.DataFrame(records)


# Calculate fairness for representative models only (one each of LR, DTree, RF, XGB, and NN).
FAIRNESS_MODELS = list(MODEL_CONFIGS.keys())

fairness_results = {}   # {model: {'credit': df, 'income': df}}

test_meta = test_t.copy()

for name in FAIRNESS_MODELS:
    y_pred_t = results[name]['temporal']['y_pred']
    y_prob_t = results[name]['temporal']['y_prob']

    fairness_results[name] = {
        'credit': compute_fairness(
            y_test_t, y_pred_t, y_prob_t,
            ['short', 'mid', 'long'], test_meta['credit_group']),
        'income': compute_fairness(
            y_test_t, y_pred_t, y_prob_t,
            ['low', 'mid', 'high'], test_meta['income_group']),
    }

print('Fairness computed for:', list(fairness_results.keys()))

In [ ]:
rows = []
for name in FAIRNESS_MODELS:
    credit_df = fairness_results[name]['credit']
    income_df = fairness_results[name]['income']
    rows.append({
        'Model':            name,
        'Temporal AUC':     round(results[name]['temporal']['auc'], 4),
        'Accuracy':         round(results[name]['temporal']['acc'], 4),  # ← 추가
        'AUC Drop':         round(results[name]['auc_drop'], 4),
        'Credit FPR_disp':  round(credit_df['FPR'].max() - credit_df['FPR'].min(), 4),
        'Credit TPR_disp':  round(credit_df['TPR'].max() - credit_df['TPR'].min(), 4),
        'Credit AUC_disp':  round(credit_df['AUC'].max() - credit_df['AUC'].min(), 4),
        'Income FPR_disp':  round(income_df['FPR'].max() - income_df['FPR'].min(), 4),
        'Income TPR_disp':  round(income_df['TPR'].max() - income_df['TPR'].min(), 4),
        'Income AUC_disp':  round(income_df['AUC'].max() - income_df['AUC'].min(), 4),
    })

df_fairness_summary = pd.DataFrame(rows).sort_values('Temporal AUC', ascending=False)
print('\n=== Fairness Disparity Summary ===')
print(df_fairness_summary.to_string(index=False))



In [ ]:
# ── Best Model: Temporal Robustness + Fairness Composite Ranking ──────────────
# Normalize each metric to 0~1, then compute weighted sum as composite score
# Lower score = better (small AUC Drop, small fairness disparity, high temporal AUC)
# ── Best Model: Temporal 강건 + Fairness 종합 랭킹 ─────────────────────────
# 각 지표를 0~1로 정규화한 뒤 가중합으로 composite score 계산
# score가 낮을수록 좋음 (AUC Drop 작고, fairness disparity 작고, temporal AUC 높음)

df = df_fairness_summary.copy()

def minmax(col):
    return (col - col.min()) / (col.max() - col.min() + 1e-9)

# Normalize (direction: lower is better)
# 정규화 (낮을수록 좋은 방향으로)
df['_auc_drop_norm']   = minmax(df['AUC Drop'])
df['_auc_perf_norm']   = minmax(-df['Temporal AUC'])
df['_acc_norm']        = minmax(-df['Accuracy'])     # higher is better → flip sign
df['_credit_fpr_norm'] = minmax(df['Credit FPR_disp'])
df['_income_fpr_norm'] = minmax(df['Income FPR_disp'])

# Weighted sum (adjust weights as needed)
# 가중합 (원하면 가중치 조절 가능)
w_robustness  = 0.4
w_performance = 0.1   # temporal AUC
w_accuracy    = 0.1   # ← add
w_fairness    = 0.4

df['composite_score'] = (
    w_robustness  * df['_auc_drop_norm'] +
    w_performance * df['_auc_perf_norm'] +
    w_accuracy    * df['_acc_norm'] +
    w_fairness    * (df['_credit_fpr_norm'] + df['_income_fpr_norm']) / 2
)

df_ranked = df[['Model', 'Temporal AUC', 'Accuracy', 'AUC Drop',
                'Credit FPR_disp', 'Income FPR_disp',
                'composite_score']].sort_values('composite_score')

print('=== Best Model Ranking (lower score = better) ===')
print(df_ranked.to_string(index=False))
print(f'\n★ Best Model: {df_ranked.iloc[0]["Model"]}')

## 4. Training & Temporal / Random AUC calculation

In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score, brier_score_loss
import time

results = {}   # {model_name: {'temporal': {...}, 'random': {...}, 'yearly': {...}}}

for name, (model, input_type) in MODEL_CONFIGS.items():
    t0 = time.time()

    if input_type == 'scaled':
        model.fit(X_train_sc, y_train_t)
        y_prob_t  = model.predict_proba(X_test_sc)[:, 1]
        y_pred_t  = model.predict(X_test_sc)
        y_prob_r  = model.predict_proba(X_rand_te_sc)[:, 1]
        y_pred_r  = model.predict(X_rand_te_sc)
    else:
        model.fit(X_train_t, y_train_t)
        y_prob_t  = model.predict_proba(X_test_t)[:, 1]
        y_pred_t  = model.predict(X_test_t)
        y_prob_r  = model.predict_proba(X_rand_te)[:, 1]
        y_pred_r  = model.predict(X_rand_te)

    # Temporal result
    results[name] = {
        'temporal': {
            'auc':   roc_auc_score(y_test_t, y_prob_t),
            'acc':   accuracy_score(y_test_t, y_pred_t),
            'brier': brier_score_loss(y_test_t, y_prob_t),
            'y_pred': y_pred_t,
            'y_prob': y_prob_t,
        },
        'random': {
            'auc':   roc_auc_score(y_rand_te, y_prob_r),
            'acc':   accuracy_score(y_rand_te, y_pred_r),
            'brier': brier_score_loss(y_rand_te, y_prob_r),
        },
        'yearly': {}
    }

    # By year AUC
    for yr, (X_yr, y_yr, X_yr_sc, _) in yearly_tests.items():
        X_in = X_yr_sc if input_type == 'scaled' else X_yr
        yp = model.predict_proba(X_in)[:, 1]
        results[name]['yearly'][yr] = roc_auc_score(y_yr, yp)

    # AUC drop
    results[name]['auc_drop'] = results[name]['random']['auc'] - results[name]['temporal']['auc']

    elapsed = time.time() - t0
    print(f"{name:20s}  temporal_AUC={results[name]['temporal']['auc']:.4f}  "
          f"random_AUC={results[name]['random']['auc']:.4f}  "
          f"AUC_drop={results[name]['auc_drop']:.4f}  ({elapsed:.1f}s)")

## 5. Fairness Calculation Function

In [ ]:
def compute_fairness(y_true, y_pred, y_prob, groups, group_col):
    """Group FPR, TPR, AUC calculation"""
    records = []
    for g in groups:
        mask = (group_col == g).values
        yt = np.array(y_true)[mask]
        yp = np.array(y_pred)[mask]
        ypr = np.array(y_prob)[mask]

        if yt.sum() == 0 or (1 - yt).sum() == 0:
            continue

        tp = ((yp == 1) & (yt == 1)).sum()
        fp = ((yp == 1) & (yt == 0)).sum()
        tn = ((yp == 0) & (yt == 0)).sum()
        fn = ((yp == 0) & (yt == 1)).sum()

        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        auc = roc_auc_score(yt, ypr)

        records.append({'group': g, 'FPR': fpr, 'TPR': tpr, 'AUC': auc, 'n': mask.sum()})
    return pd.DataFrame(records)


# Calculate fairness for representative models only (one each of LR, DTree, RF, XGB, and NN).
FAIRNESS_MODELS = ['LR_C0.01', 'LR_C10', 'DTree_d5','DTree_d7' 'DTree_d10',
                   'RF_100_d5', 'RF_100_d10', 'XGB_lr0.1', 'XGB_d6']

fairness_results = {}   # {model: {'credit': df, 'income': df}}

test_meta = test_t.copy()

for name in FAIRNESS_MODELS:
    y_pred_t = results[name]['temporal']['y_pred']
    y_prob_t = results[name]['temporal']['y_prob']

    fairness_results[name] = {
        'credit': compute_fairness(
            y_test_t, y_pred_t, y_prob_t,
            ['short', 'mid', 'long'], test_meta['credit_group']),
        'income': compute_fairness(
            y_test_t, y_pred_t, y_prob_t,
            ['low', 'mid', 'high'], test_meta['income_group']),
    }

print('Fairness computed for:', list(fairness_results.keys()))

## 6. Output Result Table

In [ ]:
# ── Overall Model Comparison Table ──────────────────────────────────────────────
rows = []
for name, res in results.items():
    rows.append({
        'Model':        name,
        'Temporal AUC': round(res['temporal']['auc'],  4),
        'Random AUC':   round(res['random']['auc'],    4),
        'AUC Drop':     round(res['auc_drop'],         4),
        'Accuracy':     round(res['temporal']['acc'],  4),
        'Brier Score':  round(res['temporal']['brier'],4),
    })

df_results = pd.DataFrame(rows).sort_values('Temporal AUC', ascending=False)
print('=== Full Model Comparison Table ===')
print(df_results.to_string(index=False))

## 7. Visulization

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Figure 1: Temporal AUC comparison (all model)
# ═══════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(14, 5))

model_names = df_results['Model'].tolist()
t_aucs = df_results['Temporal AUC'].tolist()
r_aucs = df_results['Random AUC'].tolist()

x = np.arange(len(model_names))
bars1 = ax.bar(x - 0.2, t_aucs, width=0.35, label='Temporal AUC', color='steelblue')
bars2 = ax.bar(x + 0.2, r_aucs, width=0.35, label='Random AUC',   color='lightsteelblue')

ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=30, ha='right', fontsize=9)
ax.set_ylim(0.60, 0.80)
ax.set_ylabel('AUC')
ax.set_title('Fig 1. Temporal vs Random AUC — All Models')
ax.legend()
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.3f'))
plt.tight_layout()
plt.savefig('/content/fig1_auc_comparison.png', dpi=150)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Figure 2: AUC Drop (Temporal how much decrease)
# ═══════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(14, 4))

drops = df_results['AUC Drop'].tolist()
colors = ['tomato' if d > 0 else 'seagreen' for d in drops]

ax.bar(model_names, drops, color=colors)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xticks(np.arange(len(model_names)))
ax.set_xticklabels(model_names, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('AUC Drop (Random − Temporal)')
ax.set_title('Fig 2. AUC Drop per Model\n(The higher the value, the more vulnerable it is to temporal shifts.)')
plt.tight_layout()
plt.savefig('/content/fig2_auc_drop.png', dpi=150)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Figure 3: Annual AUC Trends (Representative Model)
# ═══════════════════════════════════════════════════════════════════════
REPRESENTATIVE = ['LR_C0.01', 'LR_C10', 'DTree_d5', 'RF_100_d5', 'XGB_lr0.1']
colors_rep = ['#1f77b4', '#aec7e8', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

fig, ax = plt.subplots(figsize=(8, 5))
years = sorted(yearly_tests.keys())

for model_name, color in zip(REPRESENTATIVE, colors_rep):
    yr_aucs = [results[model_name]['yearly'].get(yr, np.nan) for yr in years]
    ax.plot(years, yr_aucs, marker='o', label=model_name, color=color, linewidth=2)

ax.set_xlabel('Test Year')
ax.set_ylabel('AUC')
ax.set_title('Fig 3. AUC over Time (train: 2007–2014)')
ax.legend(loc='lower left', fontsize=8)
ax.set_xticks(years)
plt.tight_layout()
plt.savefig('/content/fig3_auc_over_time.png', dpi=150)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Figure 4: LR Regularization Sweep — AUC + AUC Drop
# ═══════════════════════════════════════════════════════════════════════
lr_models = ['LR_C0.001', 'LR_C0.01', 'LR_C0.1', 'LR_C1', 'LR_C10']
C_vals = [0.001, 0.01, 0.1, 1, 10]
lr_t_auc  = [results[m]['temporal']['auc']  for m in lr_models]
lr_r_auc  = [results[m]['random']['auc']    for m in lr_models]
lr_drop   = [results[m]['auc_drop']         for m in lr_models]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(C_vals, lr_t_auc, marker='o', label='Temporal', color='steelblue')
axes[0].plot(C_vals, lr_r_auc, marker='s', label='Random',   color='lightsteelblue', linestyle='--')
axes[0].set_xscale('log')
axes[0].set_xlabel('C (inverse regularization strength)')
axes[0].set_ylabel('AUC')
axes[0].set_title('Fig 4a. LR: Regularization vs AUC')
axes[0].legend()

axes[1].bar(range(len(C_vals)), lr_drop, color='tomato')
axes[1].set_xticks(range(len(C_vals)))
axes[1].set_xticklabels([f'C={c}' for c in C_vals])
axes[1].set_ylabel('AUC Drop')
axes[1].set_title('Fig 4b. LR: Regularization vs AUC Drop')

plt.tight_layout()
plt.savefig('/content/fig4_lr_sweep.png', dpi=150)
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# Figure 5: Fairness — Credit History Group (Comparison of Representative Models)
# ════════════════════════════════════════════════════════════════════════════════
FAIR_MODELS_PLOT = ['LR_C0.01', 'LR_C10', 'DTree_d5', 'RF_100_d5', 'XGB_lr0.1']
credit_groups = ['short', 'mid', 'long']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics = ['FPR', 'TPR', 'AUC']
# colors_fair = ['#1f77b4', '#aec7e8', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
colors_fair = ['#1f77b4', '#aec7e8', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

for ax_i, metric in enumerate(metrics):
    x = np.arange(len(credit_groups))
    width = 0.12
    for j, (mname, col) in enumerate(zip(FAIR_MODELS_PLOT, colors_fair)):
        df_f = fairness_results[mname]['credit']
        vals = [df_f[df_f['group'] == g][metric].values[0] for g in credit_groups]
        offset = (j - len(FAIR_MODELS_PLOT) / 2) * width
        axes[ax_i].bar(x + offset, vals, width=width, label=mname, color=col)
    axes[ax_i].set_xticks(x)
    axes[ax_i].set_xticklabels(credit_groups)
    axes[ax_i].set_title(f'Fig 5{chr(97+ax_i)}. {metric} by Credit History Group')
    axes[ax_i].set_ylabel(metric)
    if ax_i == 0:
        axes[ax_i].legend(fontsize=7)

plt.tight_layout()
plt.savefig('/content/fig5_fairness_credit.png', dpi=150)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Figure 6: Fairness — Income Group (Comparison of Representative Models)
# ═══════════════════════════════════════════════════════════════════════
income_groups = ['low', 'mid', 'high']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax_i, metric in enumerate(metrics):
    x = np.arange(len(income_groups))
    for j, (mname, col) in enumerate(zip(FAIR_MODELS_PLOT, colors_fair)):
        df_f = fairness_results[mname]['income']
        vals = [df_f[df_f['group'] == g][metric].values[0] for g in income_groups]
        offset = (j - len(FAIR_MODELS_PLOT) / 2) * width
        axes[ax_i].bar(x + offset, vals, width=width, label=mname, color=col)
    axes[ax_i].set_xticks(x)
    axes[ax_i].set_xticklabels(income_groups)
    axes[ax_i].set_title(f'Fig 6{chr(97+ax_i)}. {metric} by Income Group')
    axes[ax_i].set_ylabel(metric)
    if ax_i == 0:
        axes[ax_i].legend(fontsize=7)

plt.tight_layout()
plt.savefig('/content/fig6_fairness_income.png', dpi=150)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Figure 7: Robustness–Fairness Trade-off scatter
# AUC Drop (x축) vs FPR difference across groups (y axis)
# ═══════════════════════════════════════════════════════════════════════
def fpr_disparity(model_name, group_type='credit'):
    """max FPR - min FPR across groups (fairness gap)"""
    df_f = fairness_results[model_name][group_type]
    return df_f['FPR'].max() - df_f['FPR'].min()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax_i, gtype in enumerate(['credit', 'income']):
    for mname, col in zip(FAIR_MODELS_PLOT, colors_fair):
        x_val = results[mname]['auc_drop']
        y_val = fpr_disparity(mname, gtype)
        axes[ax_i].scatter(x_val, y_val, color=col, s=100, label=mname, zorder=3)
        axes[ax_i].annotate(mname, (x_val, y_val),
                            textcoords='offset points', xytext=(5, 3), fontsize=7)
    axes[ax_i].set_xlabel('AUC Drop (Temporal shift sensitivity)')
    axes[ax_i].set_ylabel(f'FPR Disparity ({gtype} group)')
    axes[ax_i].set_title(f'Fig 7{chr(97+ax_i)}. Robustness–Fairness Trade-off\n({gtype} group)')
    axes[ax_i].axhline(0, color='gray', linestyle='--', linewidth=0.7)
    axes[ax_i].axvline(0, color='gray', linestyle='--', linewidth=0.7)

plt.tight_layout()
plt.savefig('/content/fig7_tradeoff.png', dpi=150)
plt.show()

## 8. Output Final Summary Table (For Presentation)

In [ ]:
# Representative Models Summary Table
summary_rows = []
for mname in FAIR_MODELS_PLOT:
    res = results[mname]
    credit_df = fairness_results[mname]['credit']
    income_df = fairness_results[mname]['income']

    row = {
        'Model':            mname,
        'Temporal AUC':     round(res['temporal']['auc'],  4),
        'Random AUC':       round(res['random']['auc'],    4),
        'AUC Drop':         round(res['auc_drop'],         4),
        'Brier':            round(res['temporal']['brier'],4),
        'FPR Disp (credit)':round(fpr_disparity(mname, 'credit'), 4),
        'FPR Disp (income)':round(fpr_disparity(mname, 'income'), 4),
    }
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
print('=== Summary Table (Representative Models) ===')
print(df_summary.to_string(index=False))

# CSV
df_summary.to_csv('/content/prefinal_summary.csv', index=False)
df_results.to_csv('/content/prefinal_full_results.csv', index=False)
print('\nCSV saved: prefinal_summary.csv, prefinal_full_results.csv')

## 9. Save Files (Google Drive)
Save Resulting PNG + CSV to Drive

In [ ]:
import shutil, os

SAVE_DIR = '/content/drive/MyDrive/Colab Notebooks/CSCI567/prefinal_results'
os.makedirs(SAVE_DIR, exist_ok=True)

files_to_save = [
    '/content/fig1_auc_comparison.png',
    '/content/fig2_auc_drop.png',
    '/content/fig3_auc_over_time.png',
    '/content/fig4_lr_sweep.png',
    '/content/fig5_fairness_credit.png',
    '/content/fig6_fairness_income.png',
    '/content/fig7_tradeoff.png',
    '/content/prefinal_summary.csv',
    '/content/prefinal_full_results.csv',
]

for f in files_to_save:
    if os.path.exists(f):
        shutil.copy(f, SAVE_DIR)
        print(f'Saved: {os.path.basename(f)}')
    else:
        print(f'NOT FOUND: {f}')